## Tokenization by using Bag of Word (Kaggle)

In [13]:
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import json
import autocorrect

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [14]:
# ============ LOAD DATA ============
print("Loading data from kaggle_data.json...")
with open("../cleaned_data/kaggle_data.json", "r") as f:
    kaggle_data = json.load(f)

print(f"Loaded {len(kaggle_data)} reviews")

Loading data from kaggle_data.json...
Loaded 117458 reviews


In [16]:
# ============ LEMMATIZE REVIEWS ============
print("Lemmatizing reviews with POS tagging...")
# Tag tokens of each review with POS tags to improve lemmatization
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('N'):
        return 'n'  # noun
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'  # default to noun

lemmatizer = WordNetLemmatizer()
lemmatized_reviews = []
for review in kaggle_data:
    tokens = word_tokenize(str(review["review_content"]).lower())
    pos_tags = nltk.pos_tag(tokens)
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    lemmatized_reviews.append(lemmatized_tokens)

print("Done lemmatizing reviews")
print(f"Example lemmatized reviews:")
for review in lemmatized_reviews[:5]:
    print("\t", end="")
    for token in review:
        print(token, end=" ")
    print()

Lemmatizing reviews with POS tagging...
Done lemmatizing reviews
Example lemmatized reviews:
	my friend fuck glitched out of the map . dude why 
	nobody be play this game 
	this game be n't very fun it run like as and the community be ready to kick you as soon a you make 1 mistake so do n't even waste your time unless you want the silly tf2 hat . 
	last mission be so hard 
	a an avid fan of alien shooter 2 from it heyday , i approach this game with high expectation . the core concept of the game felt reminiscent of the classic , and initially , it seem to capture that same engage essence . however , a i delve deep , several issue become apparent . the difficulty of the mission prove to be a significant challenge , and the lack of content become glaringly obvious . playing in 2024 , i encounter the unfortunate reality of a dwindling player base , which severely impact the overall experience . from a technical perspective , while the simple graphic be a nostalgic throwback that i appreci

In [17]:
# ============ REMOVE NON-ALPHANUMERIC AND STOPWORDS ============
eng_stopwords = set(stopwords.words('english'))
stopwords_removed = {}
reviews_no_stopwords = []

for review in lemmatized_reviews:
    review_no_stopwords = []
    for token in review:
        if token in eng_stopwords or not token.isalnum() or not token.isascii():
            stopwords_removed[token] = stopwords_removed.get(token, 0) + 1
        else:
            review_no_stopwords.append(token)
    reviews_no_stopwords.append(review_no_stopwords)


print(f"Removed {len(stopwords_removed)} unique stopwords and non-alphanumeric tokens")
sorted_removed = sorted(stopwords_removed.items(), key=lambda x: x[1], reverse=True)
print("Top 10 most common removed tokens:")
for i in range(min(10, len(sorted_removed))):
    print(f"\t{sorted_removed[i][0]}:\t{sorted_removed[i][1]}")

Removed 20128 unique stopwords and non-alphanumeric tokens
Top 10 most common removed tokens:
	.:	204127
	the:	179404
	,:	170428
	be:	158642
	a:	117960
	it:	111382
	and:	109450
	to:	102983
	i:	96040
	of:	78329


In [18]:
# ============ TOKENIZE REVIEWS ============
print("Tokenizing reviews...")
total_BOW = {}
reviews_BOW = []
for review in reviews_no_stopwords:
    review_dict = {}
    for token in review:
        total_BOW[token] = total_BOW.get(token, 0) + 1
        review_dict[token] = review_dict.get(token, 0) + 1
    reviews_BOW.append(review_dict)

print(f"Collected {len(total_BOW)} unique tokens")
print(f"Total token count: {sum(total_BOW.values())}")
print("Top 10 most common tokens:")
sorted_tokens = sorted(total_BOW.items(), key=lambda x: x[1], reverse=True)
for i in range(min(10, len(sorted_tokens))):
    print(f"\t{sorted_tokens[i][0]}:\t{sorted_tokens[i][1]}")

Tokenizing reviews...
Collected 46934 unique tokens
Total token count: 2289842
Top 10 most common tokens:
	game:	123969
	play:	45361
	get:	26564
	good:	25525
	like:	21140
	valve:	17219
	fun:	17160
	time:	17018
	make:	16250
	one:	15397


In [19]:
# ============ SPELL CHECK RARE TOKENS ============
rare_threshold = 8
potentially_misspelled = {word: count for word, count in total_BOW.items() if count <= rare_threshold}
print(f"Identified {len(potentially_misspelled)} potentially misspelled tokens with count <= {rare_threshold}")

print("Attempting autocorrection...")
spell = autocorrect.Speller(lang="en", fast=True)
autocorrected = {}
for word in potentially_misspelled:
    corrected = lemmatizer.lemmatize(spell(word))
    if corrected != word and corrected in total_BOW:
        total_BOW[corrected] += total_BOW[word]
        del total_BOW[word]
        autocorrected[word] = corrected        

for review in reviews_BOW:
    for word in list(review.keys()):
        if word in autocorrected:
            corrected = autocorrected[word]
            review[corrected] = review.get(corrected, 0) + review[word]
            del review[word]

print(f"Autocorrected {len(autocorrected)} tokens")
print(f"Examples:")
for misspelled in list(autocorrected.keys())[:min(10, len(autocorrected))]:
    print(f"\t{misspelled} -> {autocorrected[misspelled]}")
print(f"Remaining unique tokens: {len(total_BOW)}")

Identified 38167 potentially misspelled tokens with count <= 8
Attempting autocorrection...
Autocorrected 11468 tokens
Examples:
	tacky -> tack
	asrd -> ard
	rachael -> rachel
	mii -> mi
	brigtness -> brightness
	charakters -> character
	exlusive -> exclusive
	lole -> role
	restarts -> restart
	isngle -> single
Remaining unique tokens: 35466


In [25]:
# ============ SAVE BOW AS JSON ============
pos_count = 0

def map_pos_to_binary(rec):
    if rec == True:
        global pos_count
        pos_count += 1
        return 1
    else:
        return 0

reviews_BOW_for_json = []
for i in range(len(reviews_BOW)):
    review_with_label = {
        "recommend": map_pos_to_binary(kaggle_data[i]["is_positive"]),
        "BOW": reviews_BOW[i]
    }
    reviews_BOW_for_json.append(review_with_label)


print("Saving BOW to BOW_kaggle.json...")
with open("tokens/BOW_kaggle.json", "w") as f:
    json.dump(reviews_BOW_for_json, f, indent=4)

print(f"Positive count: {pos_count}")

print("Saving total BOW to total_BOW_kaggle.json...")
total_BOW_for_json = [{"token": token, "count": count} for token, count in total_BOW.items()]
total_BOW_for_json = sorted(total_BOW_for_json, key=lambda x: x["count"], reverse=True)
with open("tokens/total_BOW_kaggle.json", "w") as f:
    json.dump(total_BOW_for_json, f, indent=4)

Saving BOW to BOW_kaggle.json...
Positive count: 55789
Saving total BOW to total_BOW_kaggle.json...


In [21]:
# =========== ANALYZE BAD WORDS IN BOW ============
with open("../cleaned_data/bad_words.txt", "r") as f:
    bad_words = f.read().splitlines()

occuring_bad_words = [token for token in total_BOW if token in bad_words]
print(f"Total bad words in BOW: {len(occuring_bad_words)}")
print(f"Percentage of bad words in BOW: {len(occuring_bad_words) / len(total_BOW) * 100:.2f}%")
print(f"Examples of bad words in BOW:")
print(occuring_bad_words[:10])

Total bad words in BOW: 712
Percentage of bad words in BOW: 2.01%
Examples of bad words in BOW:
['fuck', 'kick', 'silly', 'shooter', 'useless', 'parasite', 'steal', 'mediocre', 'selfish', 'idiot']
